In [1]:
# -*- coding: utf-8 -*-

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import math, time
from sklearn.metrics import mean_squared_error
import plotly.express as px
import plotly.graph_objects as go
import joblib

import matplotlib.pyplot as plt
plt.rc('font', family='SimHei', size=14)  ##显示中文

import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

In [2]:
file_path = './OutputData/waterLevel.xlsx'
df = pd.read_excel(file_path)
# 将“时间”列转换为 datetime 类型，方便按时间属性筛选
# df["时间"] = pd.to_datetime(df["时间"])
# df = df[(df["时间"].dt.minute == 0) & (df["时间"].dt.second == 0)]
# df.to_excel("./OutputData/waterLevel.xlsx",index=False)

In [3]:
df

,时间,监测值
0,2025-01-01 00:00:00,766.502
1,2025-01-01 01:00:00,766.503
2,2025-01-01 02:00:00,766.512
3,2025-01-01 03:00:00,766.525
4,2025-01-01 04:00:00,766.542
...,...,...
3926,2025-07-08 22:00:00,768.085
3927,2025-07-08 23:00:00,768.082
3928,2025-07-09 00:00:00,768.100
3929,2025-07-09 01:00:00,768.111


In [4]:
# target为要输入的数据,lookback为序列长度
def split_data(target, lookback):
    data_raw = target.to_numpy() 
    data = []
    
    # you can free play（seq_length）
    for index in range(len(data_raw) - lookback + 1 ): 
        data.append(data_raw[index: index + lookback])
    
    data = np.array(data);

    # 设置测试集天数为最近30天的数据
    test_set_size = data.shape[0] * 0.1
    train_set_size = data.shape[0] - (test_set_size)
    
    x_train = data[:train_set_size,:-1,:]
    y_train = data[:train_set_size,-1,:]
    
    x_test = data[train_set_size:,:-1]
    y_test = data[train_set_size:,-1,:]
    
    return [x_train, y_train, x_test, y_test]

In [5]:
class LSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim):
        super(LSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).requires_grad_()
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).requires_grad_()
        out, (hn, cn) = self.lstm(x, (h0.detach(), c0.detach()))
        out = self.fc(out[:, -1, :]) 
        return out

In [6]:
# 模型超参数
input_dim = 1
hidden_dim = 50
num_layers = 2
output_dim = 1
lookback = 50
num_epochs = 500

# 该目录用于保存最优模型参数
os.makedirs('waterlevel_model', exist_ok=True)
# 该目录用于保存最优模型训练数据标准化格式
os.makedirs('waterlevel_scaler', exist_ok=True)


# 提取标签
target = df[['监测值']].copy()

#归一化
scaler = MinMaxScaler(feature_range=(-1, 1))
target['监测值'] = scaler.fit_transform(target['监测值'].values.reshape(-1,1))

best_scaler_path = 'waterlevel_scaler/' + 'best_scaler.pkl'
print('模型标准化格式保存路径：',best_scaler_path)

# 保存标准化参数
joblib.dump(scaler, best_scaler_path)

#划分训练集、验证集、测试集
x_train, y_train, x_test, y_test = split_data(target, lookback)

print('x_train shape:',x_train.shape)
print('y_train shape:',y_train.shape)
#print('x_val shape:',x_val.shape)
#print('y_val shape:',y_val.shape)
print('x_test shape',x_test.shape)
print('y_test shape',y_test.shape)

# 转换成张量,神经网络模型的数据要求为张量类型
x_train = torch.from_numpy(x_train).type(torch.Tensor)
x_test = torch.from_numpy(x_test).type(torch.Tensor)
#x_val = torch.from_numpy(x_val).type(torch.Tensor)
#y_val = torch.from_numpy(y_val).type(torch.Tensor)
y_train_lstm = torch.from_numpy(y_train).type(torch.Tensor)
y_test_lstm = torch.from_numpy(y_test).type(torch.Tensor)

# 模型实例化准备开始训练
model = LSTM(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_layers=num_layers)
criterion = torch.nn.MSELoss()
optimiser = torch.optim.Adam(model.parameters(), lr=0.01)

import time

hist = np.zeros(num_epochs)
best_val_loss = float('inf')  # 用于保存验证集上的最佳损失
best_test_loss = float('inf')  # 用于保存验证集上的最佳损失
best_model_path = 'waterlevel_model/' + 'best_model.pth'  # 最佳模型保存路径
#val_hist = np.zeros(num_epochs)  # 记录验证损失
test_hist = np.zeros(num_epochs)  # 记录验证损失
print('最优模型参数保存路径：',best_model_path)

start_time = time.time()
lstm = []

for t in range(num_epochs):
    # 训练模式
    model.train()
    
    y_train_pred = model(x_train)

    loss = criterion(y_train_pred, y_train_lstm)
    print("Epoch ", t, "MSE: ", loss.item())
    hist[t] = loss.item()

    optimiser.zero_grad()
    loss.backward()
    optimiser.step()

    # 验证模式
    model.eval()
    with torch.no_grad():  # 禁用梯度计算
        #y_val_pred = model(x_val)
        y_test_pred = model(x_test)
        # 反归一化
        #y_val_pred_1 = scaler.inverse_transform(y_val_pred.detach().numpy())
        #y_val_1 = scaler.inverse_transform(y_val.detach().numpy())
        y_test_pred_1 = scaler.inverse_transform(y_test_pred.detach().numpy())
        y_test_1 = scaler.inverse_transform(y_test_lstm.detach().numpy())
        test_loss = math.sqrt(mean_squared_error(y_test_1[:, 0], y_test_pred_1[:, 0]))
        print("Epoch ", t, "Test RMSE: ", test_loss)
        test_hist[t] = test_loss

        # 如果验证损失更好，则保存模型
        if test_loss < best_test_loss:
            best_test_loss = test_loss
            torch.save(model.state_dict(), best_model_path)
            print(f"Best model saved at epoch {t} with Test RMSE: {best_test_loss}")
    
training_time = time.time()-start_time
print("Training time: {}".format(training_time))

predict = pd.DataFrame(scaler.inverse_transform(y_train_pred.detach().numpy()))
original = pd.DataFrame(scaler.inverse_transform(y_train_lstm.detach().numpy()))

模型标准化格式保存路径： waterlevel_scaler/best_scaler.pkl


TypeError: slice indices must be integers or None or have an __index__ method

In [ ]:
# 模型结果可视化
sns.set_style("darkgrid")
fig = plt.figure()
fig.subplots_adjust(hspace=0.2,wspace=0.2)
plt.subplot(1,2,1)
ax = sns.lineplot(x=original.index,y=original[0],label="origin",color='royalblue')
ax = sns.lineplot(x=predict.index,y=predict[0],label="predict",color='tomato')
ax.set_title(f'predict',size=14,fontweight='bold')
ax.set_xlabel("Time",size=14)
ax.set_ylabel("Data",size=14)
ax.set_xticklabels('',size=10)

plt.subplot(1,2,2)
ax = sns.lineplot(data=hist,color='royalblue')
ax.set_xlabel("Epoch",size=14)
ax.set_ylabel("Loss",size=14)
ax.set_title("Training Loss",size=14,fontweight='bold')
fig.set_figheight(6)
fig.set_figwidth(16)
plt.show()


# 加载最优模型参数
model.load_state_dict(torch.load(best_model_path,weights_only=True))  
model.eval()  # 将模型设置为评估模式

# 加载归一化格式
scaler = joblib.load(best_scaler_path)

 # 在测试集上验证
with torch.no_grad():  # 禁用梯度计算
    y_test_pred = model(x_test)
    y_train_pred = model(x_train)

# 反归一化
y_train_pred = scaler.inverse_transform(y_train_pred.detach().numpy())
y_train = scaler.inverse_transform(y_train_lstm.detach().numpy())
#y_val = scaler.inverse_transform(y_val.detach().numpy())
#y_val_pred = scaler.inverse_transform(y_val_pred.detach().numpy())
y_test_pred = scaler.inverse_transform(y_test_pred.detach().numpy())
y_test = scaler.inverse_transform(y_test_lstm.detach().numpy())

# 计算MSE
trainScore = math.sqrt(mean_squared_error(y_train[:,0], y_train_pred[:,0]))
print('Train Score: %.2f RMSE' % (trainScore))
#valScore = math.sqrt(mean_squared_error(y_val[:, 0], y_val_pred[:, 0]))
#print('Validation Score: %.2f RMSE' % (valScore))
testScore = math.sqrt(mean_squared_error(y_test[:,0], y_test_pred[:,0]))
print('Test Score: %.2f RMSE' % (testScore))
lstm.append(trainScore)
#lstm.append(valScore)
lstm.append(testScore)
lstm.append(training_time)


In [ ]:
# 制作可视化数据集
# 对训练预测数据进行偏移处理
# 制作可视化数据集
trainPredictPlot = np.empty_like(target)
trainPredictPlot[:, :] = np.nan
trainPredictPlot[lookback - 1:len(y_train_pred)+lookback - 1, :] = y_train_pred

# shift test predictions for plotting
testPredictPlot = np.empty_like(target)
testPredictPlot[:, :] = np.nan
testPredictPlot[len(y_train_pred)+lookback-1:len(target), :] = y_test_pred

original = scaler.inverse_transform(target['监测值'].values.reshape(-1,1))

# 合并所有数据
predictions = np.append(trainPredictPlot, testPredictPlot, axis=1)
predictions = np.append(predictions, original, axis=1)
result = pd.DataFrame(predictions)

# 将集合好的result画图可视化
fig = go.Figure()
fig.add_trace(go.Scatter(go.Scatter(x=result.index, y=result[0],
                    mode='lines',
                    name='Train prediction')))
fig.add_trace(go.Scatter(x=result.index, y=result[1],
                    mode='lines',
                    name='Test prediction'))
fig.add_trace(go.Scatter(go.Scatter(x=result.index, y=result[2],
                    mode='lines',
                    name='Actual Value')))
fig.update_layout(
        title={
        'text': f'predict',  # 标题
        'x': 0.5,  # 标题水平居中
        'xanchor': 'center',
        'y': 0.9,  # 标题在图表的顶部
        'yanchor': 'top',
        'font': {
            'family': "Rockwell",
            'size': 20,
            'color': 'white'
        }
    },
    xaxis=dict(
        showline=True,
        showgrid=True,
        showticklabels=False,
        linecolor='white',
        linewidth=2
    ),
    yaxis=dict(
        title_text="Value",
        titlefont=dict(
            family='Rockwell',
            size=12,
            color='white',
        ),
        showline=True,
        showgrid=True,
        showticklabels=True,
        linecolor='white',
        linewidth=2,
        ticks='outside',
        tickfont=dict(
            family='Rockwell',
            size=12,
            color='white',
        ),
    ),
    showlegend=True,
    template = 'plotly_dark'

)

annotations = []
annotations.append(dict(xref='paper', yref='paper', x=0.0, y=1.05,
                              xanchor='left', yanchor='bottom',
                              text='Results (LSTM)',
                              font=dict(family='Rockwell',
                                        size=26,
                                        color='white'),
                              showarrow=False))
fig.update_layout(annotations=annotations)
fig.show()

In [ ]:
# 单步滚动预测函数 - 根据需求实现
def rolling_forecast_with_update(model, scaler, original_data, lookback, forecast_steps):
    """
    使用单步LSTM模型实现滚动预测并更新原数据集
    
    参数:
    model: 训练好的LSTM模型
    scaler: 用于归一化和反归一化的缩放器
    original_data: 原始数据集 (DataFrame，包含时间和监测值两列)
    lookback: 模型输入序列长度
    forecast_steps: 需要预测的未来步数
    
    返回:
    updated_data: 包含预测值的更新后的数据集
    """
    model.eval()
    
    # 确保数据格式正确
    time_column = original_data.columns[0]  # 假设第一列是时间
    value_column = original_data.columns[1]  # 假设第二列是监测值
    
    # 复制原始数据，避免修改原始数据
    data = original_data.copy()
    
    # 提取最后lookback个时间步的数据作为初始输入
    last_sequence = data[value_column].values[-lookback:].reshape(1, lookback, 1)
    
    # 归一化
    last_sequence_scaled = scaler.transform(last_sequence.reshape(-1, 1)).reshape(1, lookback, 1)
    
    # 转换为张量
    last_sequence_tensor = torch.FloatTensor(last_sequence_scaled)
    
    # 预测未来steps步
    with torch.no_grad():
        for i in range(forecast_steps):
            # 预测下一个时间步
            pred = model(last_sequence_tensor)
            pred_value = pred.item()
            
            # 反归一化预测值
            pred_value_actual = scaler.inverse_transform([[pred_value]])[0, 0]
            
            # 生成下一个时间戳（假设时间间隔均匀）
            last_time = pd.to_datetime(data[time_column].iloc[-1])
            time_delta = pd.to_datetime(data[time_column].iloc[-1]) - pd.to_datetime(data[time_column].iloc[-2])
            next_time = last_time + time_delta
            
            # 创建新的预测行
            new_row = pd.DataFrame({time_column: [next_time], value_column: [pred_value_actual]})
            
            # 将新行添加到数据集中
            data = pd.concat([data, new_row], ignore_index=True)
            
            # 更新输入序列：移除第一个时间步，添加新预测值
            new_sequence = np.roll(last_sequence_scaled, -1, axis=1)
            new_sequence[0, -1, 0] = pred_value
            last_sequence_scaled = new_sequence
            last_sequence_tensor = torch.FloatTensor(last_sequence_scaled)
    
    return data

In [ ]:
data = rolling_forecast_with_update(model, scaler, df, lookback, 10)

In [ ]:
data.iloc[-10:,:]